In [ ]:
import torch
import torch.nn as nn
import numpy as np
import math
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

torch.manual_seed(42)
np.random.seed(42)

class BlackBoxDatasetGenerator:
    def __init__(self, ode_func, u0, t_span=(0.0, 10.0), num_points=500, noise_level=0.001):
        self.ode_func = ode_func
        self.u0 = np.array(u0, dtype=float)
        self.t_span = t_span
        self.num_points = num_points
        self.noise_level = noise_level
        
    def generate(self):
        t_eval = np.linspace(self.t_span[0], self.t_span[1], self.num_points)
        def wrapper(t, u):
            try:
                res = self.ode_func(t, u)
            except TypeError:
                res = self.ode_func(u)
            return np.array(res).flatten()

        sol = solve_ivp(wrapper, self.t_span, self.u0, t_eval=t_eval, method='RK45', rtol=1e-8, atol=1e-8)
        t_data = torch.tensor(sol.t, dtype=torch.float32).reshape(-1, 1)
        u_true = torch.tensor(sol.y.T, dtype=torch.float32)
        u_data = u_true + self.noise_level * torch.randn_like(u_true)
        dt = (t_data[1] - t_data[0]).item()
        return t_data, u_data, dt


class FourierFeatureLayer(nn.Module):
    def __init__(self, in_dim, mapping_size=32, scale=2.5):
        super().__init__()
        self.B = nn.Parameter(torch.randn(in_dim, mapping_size) * scale, requires_grad=False)
        self.out_dim = mapping_size * 2

    def forward(self, x):
        proj = torch.matmul(x, self.B)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


class StableTrajectoryPINN(nn.Module):
    def __init__(self, dim=2, taylor_order=2, hidden_dim=64, fourier_scale=2.5):
        super().__init__()
        self.dim = dim
        self.taylor_order = taylor_order
        self.fourier_encoder = FourierFeatureLayer(in_dim=dim, mapping_size=32, scale=fourier_scale)
        encoded_dim = self.fourier_encoder.out_dim
        
        # Daha geniş ve kapasiteli gizli katmanlar
        self.a_net = nn.Sequential(nn.Linear(encoded_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, dim))
        self.b_net = nn.Sequential(nn.Linear(encoded_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, dim))
        
    def forward(self, u):
        u_enc = self.fourier_encoder(u)
        return self.a_net(u_enc) + self.b_net(u_enc)
        
    def compute_taylor_step(self, u, dt):
        u = u.clone().detach().requires_grad_(True)
        f = self.forward(u)
        taylor_sum = u + dt * f
        u_deriv = f
        
        for k in range(2, self.taylor_order + 1):
            grad_outputs = torch.ones_like(u_deriv)
            d_deriv_du = torch.autograd.grad(
                outputs=u_deriv, inputs=u, grad_outputs=grad_outputs, 
                create_graph=True, retain_graph=True
            )[0]
            u_deriv = d_deriv_du * f
            term = (dt ** k) / math.factorial(k) * u_deriv
            taylor_sum = taylor_sum + term
            
        return taylor_sum

    def fit_stable(self, u_data, dt, epochs=2000, lr=0.003, rollout_steps=10, batch_size=32):
        optimizer = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=1e-6)
        total_points = len(u_data)
        max_start_idx = total_points - rollout_steps - 1
        
        ema_weights = {name: param.clone().detach() for name, param in self.named_parameters()}
        ema_decay = 0.995
        
        print(f"\nTraining Stable PINN (Points: {total_points})...")
        for epoch in range(epochs):
            optimizer.zero_grad()
            start_indices = torch.randint(0, max_start_idx, (batch_size,))
            curr = u_data[start_indices].clone()
            total_loss = 0.0
            
            for step in range(rollout_steps):
                target_indices = start_indices + step + 1
                target_next = u_data[target_indices]
                with torch.enable_grad():
                    curr = self.compute_taylor_step(curr, dt)
                total_loss = total_loss + torch.mean((curr - target_next) ** 2)
                
            loss = total_loss / rollout_steps
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=0.2) # Daha sıkı gradyan kırpma (Exploding gradient önleyici)
            optimizer.step()
            
            with torch.no_grad():
                for name, param in self.named_parameters():
                    ema_weights[name].mul_(ema_decay).add_(param.data, alpha=1.0 - ema_decay)
            
            if (epoch + 1) % 400 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.6e}")

        with torch.no_grad():
            for name, param in self.named_parameters():
                param.data.copy_(ema_weights[name])
        print("-> Stable EMA weights applied!")


def extract_direct_neural_formula(model):
    """
    Formülü sadece tek bir merkez noktasına lineerleştirmek yerine, 
    doğrudan sinir ağının orijinal non-linear yapısını (Fourier + Tanh) SymPy'a aktarır.
    Bu sayede test bölgesinde sapma (drift) yapmaz.
    """
    model.eval()
    x0 = sp.Symbol('x0', real=True)
    x1 = sp.Symbol('x1', real=True)
    
    B = model.fourier_encoder.B.detach().numpy()
    proj_sym = [B[0, j] * x0 + B[1, j] * x1 for j in range(B.shape[1])]
    fourier_out = [sp.sin(p) for p in proj_sym] + [sp.cos(p) for p in proj_sym]
    
    def process_subnetwork(net):
        W1 = net[0].weight.detach().numpy()
        b1 = net[0].bias.detach().numpy()
        W2 = net[2].weight.detach().numpy()
        b2 = net[2].bias.detach().numpy()
        
        hidden = []
        for i in range(W1.shape[0]):
            lin_expr = b1[i]
            for k in range(len(fourier_out)):
                lin_expr += W1[i, k] * fourier_out[k]
            hidden.append(sp.tanh(lin_expr))
            
        output = []
        for i in range(W2.shape[0]):
            out_expr = b2[i]
            for k in range(len(hidden)):
                out_expr += W2[i, k] * hidden[k]
            output.append(out_expr)
        return output

    a_out = process_subnetwork(model.a_net)
    b_out = process_subnetwork(model.b_net)
    f_sympy = [a_out[i] + b_out[i] for i in range(len(a_out))]
    
    print("\n" + "="*75)
    print("      DIRECT NON-LINEAR SYMPY FORMULA (Drift-Free Extrapolation)")
    print("="*75)
    sp.pprint(sp.Eq(sp.Symbol('dx0/dt'), f_sympy[0]))
    print("-" * 75)
    sp.pprint(sp.Eq(sp.Symbol('dx1/dt'), f_sympy[1]))
    print("="*75)
    
    return f_sympy, x0, x1


def evaluate_formula_relative(f_sympy, x0_sym, x1_sym, t_range, u_true_range, label="TEST"):
    f_numeric_0 = sp.lambdify((x0_sym, x1_sym), f_sympy[0], 'numpy')
    f_numeric_1 = sp.lambdify((x0_sym, x1_sym), f_sympy[1], 'numpy')
    
    def ode_wrapper(t, u):
        return [float(f_numeric_0(u[0], u[1])), float(f_numeric_1(u[0], u[1]))]
    
    t_span = (float(t_range[0]), float(t_range[-1]))
    t_eval = t_range.numpy().flatten()
    u0_start = u_true_range[0].numpy()
    
    sol = solve_ivp(ode_wrapper, t_span, u0_start, t_eval=t_eval, method='RK45')
    u_pred = sol.y.T
    u_true = u_true_range.numpy()
    
    print(f"\n" + "="*75)
    print(f"          {label} RANGE PERFORMANCE REPORT (Relative Metrics)")
    print("="*75)
    
    for i in range(2):
        mse = np.mean((u_pred[:, i] - u_true[:, i]) ** 2)
        rmse = np.sqrt(mse)
        data_std = np.std(u_true[:, i])
        nrmse = rmse / (data_std + 1e-8)
        
        ss_res = np.sum((u_true[:, i] - u_pred[:, i]) ** 2)
        ss_tot = np.sum((u_true[:, i] - np.mean(u_true[:, i])) ** 2)
        r2 = 1.0 - (ss_res / (ss_tot + 1e-8))
        
        print(f"Dimension {i} (x{i}):")
        print(f"  -> RMSE  : {rmse:.5f} (Data Std: {data_std:.3f})")
        print(f"  -> NRMSE : {nrmse:.4f} ({nrmse*100:.2f}% relative error)")
        print(f"  -> R²    : {r2:.4f} (1.0 = Perfect Fit)")
        print("-" * 75)
        
    print("="*75)
    return u_pred


if __name__ == "__main__":
    def target_system(u):
        x0, x1 = u[0], u[1]
        dx0 = -0.5 * x0 + np.sin(x1)
        dx1 = -0.5 * x1 + np.cos(x0)
        return [dx0, dx1]

    generator = BlackBoxDatasetGenerator(ode_func=target_system, u0=[1.5, 1.0], t_span=(0.0, 10.0), num_points=500)
    t_data, u_data, dt = generator.generate()
    
    split_idx = int(len(t_data) * 0.7)
    t_train, u_train = t_data[:split_idx], u_data[:split_idx]
    t_test, u_test = t_data[split_idx:], u_data[split_idx:]

    model = StableTrajectoryPINN(dim=2, taylor_order=2, hidden_dim=64, fourier_scale=2.5)
    model.fit_stable(u_train, dt, epochs=2000, lr=0.003, rollout_steps=10, batch_size=32)

    f_sympy, x0_sym, x1_sym = extract_direct_neural_formula(model)

    u_train_pred = evaluate_formula_relative(f_sympy, x0_sym, x1_sym, t_train, u_train, label="TRAIN")
    u_test_pred = evaluate_formula_relative(f_sympy, x0_sym, x1_sym, t_test, u_test, label="TEST")

    # Görselleştirme
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    t_test_np = t_test.numpy().flatten()
    
    axes[0].plot(t_test_np, u_test[:, 0].numpy(), 'go', label='True Test X0', alpha=0.6)
    axes[0].plot(t_test_np, u_test_pred[:, 0], 'r-', label='Direct Formula X0', linewidth=2)
    axes[0].set_title('Dimension 0 Test Prediction (Drift-Free)')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    axes[1].plot(u_test[:, 0].numpy(), u_test[:, 1].numpy(), 'g--', label='True Test Manifold', alpha=0.7)
    axes[1].plot(u_test_pred[:, 0], u_test_pred[:, 1], 'r-', label='Formula Manifold', linewidth=2)
    axes[1].set_title('Phase Space Trajectory')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()